# Section 1 : Environment Setup & Imports
Installing the required dependencies and load API keys.

## 1.1 installing dependencies

In [1]:
import warnings
warnings.filterwarnings('ignore')

# !pip install -r requirements.txt

## 1.2 Loading the OpenAI API key

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

# Section 2 : Data Chunking
loading the mock data and split it into chunks.

In [3]:
from rag_utils import DataProcessor

# Initialize the Data processor
processor = DataProcessor()

# Load and split the document
chunks = processor.load_and_split(directory_path="data/", chunk_size=500, chunk_overlap=100)

print(f"Total chunks created: {len(chunks)}")

sources = set([chunk.metadata.get("source") for chunk in chunks])
print(f"Sources loaded: {sources}")

100%|██████████| 1/1 [00:00<00:00, 475.28it/s]

Total chunks created: 45
Sources loaded: {'data/linkedin_profile.txt', 'data/portfolio.txt', 'data/github_projects.md', 'data/certifications.txt', 'data/Mohammad_CV.pdf'}


# Section 3 : Vector Database & Retriever
Convert the text chunks into embeddings using HuggingFace's `all-MiniLM-L6-v2` model and store them in a ChromaDB vector store.

In [4]:
# Create vector store
vectorstore = processor.create_vector_store(chunks, persist_directory="./chroma_db")
# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Section 4 : Building the RAG Pipeline
inject our retriever into the `RAGPipeline` class. This class handles the prompt engineering and the LangChain Expression Language (LCEL) to ensure the LLM answers based *only* on the retrieved context and cites its sources.

In [5]:
from rag_chain import RAGPipeline

# Initialize the RAG pipeline with the retriever
rag_pipeline = RAGPipeline(retriever=retriever)
rag_chain = rag_pipeline.build()

# Section 5 : Inference

In [6]:
def query_rag(question: str):
    print(f"Question: {question}\n")

    # Invoke the chain
    result = rag_chain.invoke(question)

    print(f"Answer:\n{result['answer']}\n")

    print("Source Documents Used:")
    for i, doc in enumerate(result["context"]):
        # Print the source file name along with the content
        source_name = doc.metadata.get("source", "Unknown")
        print(f"  [Source {i+1} - {source_name}]:")
        print(f"  {doc.page_content.strip()}\n")
    print("-" * 50)

## Test 1: Specific skill question (Tests Skills & Languages section)

In [7]:
query_rag("What are Mohammad's main programming languages?")

Question: What are Mohammad's main programming languages?



OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1791417600000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.', 'provider_name': None}}, 'user_id': 'user_3KMCvyVQv7jUxG3AYe4aZvOfkjK'}

## Test 2: Experience question (Tests Professional Experience section)

In [ ]:
query_rag("Where did Mohammad work and what did he do there?")

## Test 3: GitHub question (Tests Projects section)

In [ ]:
query_rag("What kind of projects has Mohammad worked on?")

## Test 4: Certifications question (Tests Certifications section)

In [ ]:
query_rag("What certifications does Mohammad have?")

## Test 5: Out-of-scope question (testing hallucination guardrails)

In [ ]:
query_rag("What is Mohammad's favorite food?")